<a href="https://colab.research.google.com/github/sakethnandam/exoplanet-detection/blob/main/autoencoder_data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q lightkurve astroquery pandas numpy matplotlib scikit-learn torch torchvision tqdm

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

import lightkurve as lk

from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.model_selection import train_test_split

np.random.seed(42)
torch.manual_seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
confirmed_planets = NasaExoplanetArchive.query_criteria(
    table="pscomppars",
    select="pl_name,hostname,disc_facility,pl_rade,pl_bmasse",
    where="disc_facility like '%Kepler%' or disc_facility like '%TESS%'"
)

confirmed_df = confirmed_planets.to_pandas()
print(f"Found {len(confirmed_df)} confirmed exoplanets from Kepler/TESS")
print("\nSample of confirmed planets:")
print(confirmed_df.head())

In [ ]:
#getting kepler objects of interest
print("\nDownloading Kepler Objects of Interest (KOIs)...")

kois = NasaExoplanetArchive.query_criteria(
    table="cumulative",
    select="kepoi_name,kepid,koi_disposition,koi_period,koi_depth,koi_duration"
)

koi_df = kois.to_pandas()
print(f"Found {len(koi_df)} KOIs total")

confirmed_kois = koi_df[koi_df['koi_disposition'] == 'CONFIRMED']
candidate_kois = koi_df[koi_df['koi_disposition'] == 'CANDIDATE']
false_positive_kois = koi_df[koi_df['koi_disposition'] == 'FALSE POSITIVE']

print(f"\nBreakdown:")
print(f"  Confirmed: {len(confirmed_kois)}")
print(f"  Candidates: {len(candidate_kois)}")
print(f"  False Positives: {len(false_positive_kois)}")

In [ ]:
#data config
SAMPLE_SIZE_PER_CLASS = 50  #small for testing, increase later
WINDOW_SIZE = 512
MAX_WINDOWS_PER_STAR = 10

print(f"Configuration:")
print(f"  Sample size per class: {SAMPLE_SIZE_PER_CLASS}")
print(f"  Window size: {WINDOW_SIZE}")
print(f"  Max windows per star: {MAX_WINDOWS_PER_STAR}")

In [ ]:
def download_light_curve_safe(target_id, mission='Kepler'):
    try:
        search_result = lk.search_lightcurve(f"{mission.upper()} {target_id}", mission=mission)

        if len(search_result) == 0:
            return None

        lc = search_result[0].download()

        if lc is None:
            return None

        return lc

    except Exception as e:
        return None


def preprocess_light_curve(lc):
    try:
        lc = lc.remove_nans()
        lc = lc.normalize()
        lc = lc.flatten(window_length=301)
        flux = lc.flux.value.astype(np.float32)

        if len(flux) < WINDOW_SIZE:
            return None

        return flux

    except Exception as e:
        return None


def create_windows(flux, window_size=WINDOW_SIZE, max_windows=MAX_WINDOWS_PER_STAR):
    windows = []

    for i in range(0, len(flux) - window_size + 1, window_size):
        if len(windows) >= max_windows:
            break

        window = flux[i:i + window_size]
        median = np.median(window)
        if median > 0:
            window = window / median - 1.0
            windows.append(window)

    return windows

In [ ]:
def collect_labeled_data(koi_df, label, sample_size, mission='Kepler'):
    data = []

    sampled_df = koi_df.sample(min(sample_size * 3, len(koi_df)), replace=False)

    successful = 0

    print(f"\nCollecting {label} data...")

    for idx, row in tqdm(sampled_df.iterrows(), total=len(sampled_df), desc=f"Downloading {label}"):
        if successful >= sample_size:
            break

        kepid = row['kepid']

        lc = download_light_curve_safe(kepid, mission=mission)

        if lc is None:
            continue

        flux = preprocess_light_curve(lc)

        if flux is None:
            continue

        windows = create_windows(flux)

        if len(windows) == 0:
            continue

        for window in windows:
            metadata = {
                'kepid': kepid,
                'kepoi_name': row.get('kepoi_name', ''),
                'period': row.get('koi_period', np.nan),
                'depth': row.get('koi_depth', np.nan),
                'duration': row.get('koi_duration', np.nan)
            }
            data.append((window, label, metadata))

        successful += 1

    print(f"Successfully collected {len(data)} windows from {successful} stars for {label}")

    return data

In [ ]:
#collecting data for positive, candidiates, false positives
print("="*60)
print("Starting data collection...")
print("="*60)

confirmed_data = collect_labeled_data(
    confirmed_kois,
    'confirmed',
    SAMPLE_SIZE_PER_CLASS,
    mission='Kepler'
)

candidate_data = collect_labeled_data(
    candidate_kois,
    'candidate',
    SAMPLE_SIZE_PER_CLASS,
    mission='Kepler'
)

false_positive_data = collect_labeled_data(
    false_positive_kois,
    'false_positive',
    SAMPLE_SIZE_PER_CLASS,
    mission='Kepler'
)

all_data = confirmed_data + candidate_data + false_positive_data

print("\n" + "="*60)
print(f"Total data collected: {len(all_data)} windows")
print(f"  Confirmed: {len(confirmed_data)}")
print(f"  Candidates: {len(candidate_data)}")
print(f"  False Positives: {len(false_positive_data)}")
print("="*60)

In [ ]:
#create metadata for analysis
metadata_list = []
for window, label, metadata in all_data:
    metadata['label'] = label
    metadata_list.append(metadata)

metadata_df = pd.DataFrame(metadata_list)

#saving metadata to .csv
metadata_df.to_csv('exoplanet_metadata.csv', index=False)
print("\nMetadata saved to 'exoplanet_metadata.csv'")
print("\nMetadata summary:")
print(metadata_df.groupby('label').size())